In [ ]:
import pandas as pd

def simulador_completo_estrategia(piloto, custo_pit_nosso, custo_pit_adversario, deg_por_volta, ritmo_atual_adversario, ritmo_esperado_pneu_novo, penalidade_pneu_frio):
    print(f"SIMULADOR DE ESPECTRO COMPLETO: {piloto}")
    print("-" * 75)
    
    resultados = []
    
    # Delta Box: O que ganhamos ou perdemos na operação dos mecânicos
    delta_box = custo_pit_adversario - custo_pit_nosso
    diferenca_ritmo = ritmo_atual_adversario - ritmo_esperado_pneu_novo
    
    # 1. CENÁRIO: UNDERCUT (Parar ANTES do adversário)
    for voltas in range(3, 0, -1):
        tempo_perdido_adversario = sum([deg_por_volta * i for i in range(1, voltas + 1)])
        ganho_bruto_pista = diferenca_ritmo * voltas
        ganho_pista_liquido = (ganho_bruto_pista + tempo_perdido_adversario) - penalidade_pneu_frio
        ganho_total = ganho_pista_liquido + delta_box
        
        resultados.append({
            'Estratégia': f'Undercut ({voltas}v antes)',
            'Ganho Pista (s)': round(ganho_pista_liquido, 3),
            'Delta Box (s)': round(delta_box, 3),
            'Ganho TOTAL (s)': round(ganho_total, 3),
            'Veredito': "✅ FUNCIONA" if ganho_total > 0 else "❌ FALHA"
        })
        
    # Linha divisória neutra (Parar na mesma volta)
    resultados.append({'Estratégia': '--- MESMA VOLTA ---', 'Ganho Pista (s)': 0.0, 'Delta Box (s)': round(delta_box, 3), 'Ganho TOTAL (s)': round(delta_box, 3), 'Veredito': "IGUAL" if delta_box == 0 else ("✅ GANHA NO BOX" if delta_box > 0 else "❌ PERDE NO BOX")})

    # 2. CENÁRIO: OVERCUT (Parar DEPOIS do adversário)
    for voltas in range(1, 4):
        # Nossa degradação por ficar na pista com pneu velho
        deg_acumulada_nossa = sum([deg_por_volta * i for i in range(1, voltas + 1)])
        
        # Nosso ganho de pista = Adversário escorregou no pneu frio - Vantagem de ritmo dele - Nossa degradação
        ganho_pista_overcut = penalidade_pneu_frio - (diferenca_ritmo * voltas) - deg_acumulada_nossa
        ganho_total_overcut = ganho_pista_overcut + delta_box
        
        resultados.append({
            'Estratégia': f'Overcut ({voltas}v depois)',
            'Ganho Pista (s)': round(ganho_pista_overcut, 3),
            'Delta Box (s)': round(delta_box, 3),
            'Ganho TOTAL (s)': round(ganho_total_overcut, 3),
            'Veredito': "✅ FUNCIONA" if ganho_total_overcut > 0 else "❌ FALHA"
        })
        
    df_simulacao = pd.DataFrame(resultados)
    print(df_simulacao.to_string(index=False))
    print("-" * 75)

# --- ÁREA DE EXECUÇÃO: O VERDADEIRO CENÁRIO DO IBIAPINA (CARRO 80) ---
simulador_completo_estrategia(
    piloto="Alfredinho Ibiapina (Carro 80)",
    custo_pit_nosso=37.851,          # O pit real do carro 80 (muito lento)
    custo_pit_adversario=33.247,     # Assumindo que o adversário teve um pit normal
    deg_por_volta=0.050,             # A degradação do Ibiapina no Stint 2
    ritmo_atual_adversario=82.500,
    ritmo_esperado_pneu_novo=81.800,
    penalidade_pneu_frio=1.200
)

print("\n") # Dá um "Enter" a mais no terminal para separar visualmente as tabelas

# --- ÁREA DE EXECUÇÃO: O CENÁRIO DO ZONTA ---
simulador_completo_estrategia(
    piloto="Ricardo Zonta (Carro 10)",
    custo_pit_nosso=33.247,          # O Pit normal do Zonta
    custo_pit_adversario=33.247,     # Assumindo que o adversário não errou no box
    deg_por_volta=0.031,             # A degradação que calculamos para o Zonta
    ritmo_atual_adversario=82.500,
    ritmo_esperado_pneu_novo=81.800,
    penalidade_pneu_frio=1.200
)

SIMULADOR DE ESPECTRO COMPLETO: Alfredinho Ibiapina (Carro 80)
---------------------------------------------------------------------------
         Estratégia  Ganho Pista (s)  Delta Box (s)  Ganho TOTAL (s)       Veredito
Undercut (3v antes)             1.20         -4.604           -3.404        ❌ FALHA
Undercut (2v antes)             0.35         -4.604           -4.254        ❌ FALHA
Undercut (1v antes)            -0.45         -4.604           -5.054        ❌ FALHA
--- MESMA VOLTA ---             0.00         -4.604           -4.604 ❌ PERDE NO BOX
Overcut (1v depois)             0.45         -4.604           -4.154        ❌ FALHA
Overcut (2v depois)            -0.35         -4.604           -4.954        ❌ FALHA
Overcut (3v depois)            -1.20         -4.604           -5.804        ❌ FALHA
---------------------------------------------------------------------------


SIMULADOR DE ESPECTRO COMPLETO: Ricardo Zonta (Carro 10)
------------------------------------------------------